**Running this on Google Colab.** The first code cell fetches this entry's files and installs the packages it was run with, pinned for Colab's Python version. The results in the registry come from the entry's own environment ([requirements.lock.txt](requirements.lock.txt), Python 3.13); a different Python or platform can change the last digits of some numbers.

In [ ]:
# On Google Colab: fetch this entry's files and install its pinned packages. Elsewhere this cell does nothing.
import sys
if "google.colab" in sys.modules:
    import os, subprocess
    if not os.path.isdir("/content/encore"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", "colab", "--filter=blob:none", "--sparse",
                        "https://github.com/encorescience/encore-registry", "/content/encore"], check=True)
        subprocess.run(["git", "-C", "/content/encore", "sparse-checkout", "set", "entries/tingley-2019"], check=True)
    os.chdir("/content/encore/entries/tingley-2019")
    sys.path.insert(0, os.getcwd())
    version = f"{sys.version_info.major}.{sys.version_info.minor}"
    lock = f"requirements.colab-{version}.txt"
    if not os.path.exists(lock):
        print(f"No lock for Python {version}: installing the entry's unpinned requirements (requirements.in).")
        lock = "requirements.in"
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
    subprocess.run(["uv", "pip", "install", "--system", "-q", "-r", lock], check=True)
    print("Installed. If Colab asks to restart the session, restart it and run the notebook from the top again.")


# Earlier findings of tingley-2019-01, checked on DANDI:000218 version 0.261110.1911

Dataset: DANDI:000218, version 0.261110.1911 (doi:10.48324/dandi.000218/0.261110.1911), streamed from the archive.

The earlier attempt, tingley-2019-01, tested version 0.250624.0439 and recorded 11 findings. This notebook is
written by the reconcile step of the repeat attempt tingley-2019-02 (docs/protocol.md, "Repeat attempts"). It has
one short section per earlier finding, each opening with the earlier statement. Where a finding of
tingley-2019-02 reports the same problem, the section says which and prints the counts on this version that
show how much of it remains; otherwise the section checks the earlier finding directly on this version.
No outcome of either attempt rests on this notebook.

## Setup

The metadata of all 98 files are streamed (electrodes, units with their raw spike times, the event tables and
their descriptions); the LFP is read only in the last section, for two channels of each recording there.

In [ ]:
import collections, json, time, urllib.request, zlib
from concurrent.futures import ThreadPoolExecutor
import numpy as np, pandas as pd
import tingley2019 as T
pd.set_option("display.width", 200); pd.set_option("display.max_rows", 200)
KEY = "tingley-2019/earlier-findings"

def meta(row):
    """What the checks need from one file. Spike times are kept as stored, not sorted."""
    f = T.open_nwb(row["url"])
    m = dict(path=row["path"], subject=row["subject"], session=row["session"], url=row["url"])
    m["nwb_subject"] = T._s(f["general/subject/subject_id"][()])
    el = f["general/extracellular_ephys/electrodes"]
    m["el_cols"] = sorted(el.keys())
    m["el_region"] = np.array([T._s(x).lower() for x in el["region"][:]])
    m["ca1_flag"], m["ca3_flag"] = el["ca1_channel"][:].astype(bool), el["ca3_channel"][:].astype(bool)
    m["det"] = [T._s(x) for x in el["ripple_detection_channel"][:]]
    m["unit_cols"], m["inversions"], m["unit_region"], m["n_units"] = [], [], np.array([], str), 0
    m["wf_units"], m["iso_units"], m["units_desc"] = 0, 0, ""
    if "units" in f:
        u = f["units"]
        m["unit_cols"] = sorted(u.keys())
        st, idx = u["spike_times"][:], u["spike_times_index"][:]
        a = np.r_[0, idx[:-1]]
        m["inversions"] = [int((np.diff(st[i:j]) < 0).sum()) for i, j in zip(a, idx)]
        m["unit_region"] = np.array([T._s(x).lower() for x in u["region"][:]])
        m["n_units"] = len(idx)
        m["units_desc"] = T._s(u.attrs.get("description", ""))
        if "raw_waveform" in u:
            w = u["raw_waveform"][:]
            m["wf_units"] = int(np.isfinite(w.reshape(len(w), -1)).all(1).sum())
        if "isolation_distance" in u:
            m["iso_units"] = int(np.isfinite(u["isolation_distance"][:]).sum())
    m["events"], m["event_desc"] = {}, {}
    for k, g in f["processing/ecephys"].items():
        if T._s(g.attrs.get("neurodata_type", "")) == "TimeIntervals":
            m["events"][k] = np.sort(g["peak_time"][:])
            m["event_desc"][k] = T._s(g.attrs.get("description", ""))
    f.close()
    return m

t0 = time.time()
A = T.list_assets()
with ThreadPoolExecutor(8) as ex:
    M = list(ex.map(meta, [r for _, r in A.iterrows()]))
print(f"{len(M)} files of dandiset {T.DANDISET} version {T.VERSION} read in {time.time() - t0:.0f} s")
print("files by subject:", dict(collections.Counter(m["subject"] for m in M)))
print("units:", sum(m["n_units"] for m in M), "| electrodes:", sum(len(m["el_region"]) for m in M))
print("event tables (files holding each):", dict(collections.Counter(k for m in M for k in m["events"])))

## Finding 1

> The files hold no detected LS HFOs or hippocampal ripples, nor the detection channels and parameters, which
> the Methods say are stored with each event structure. Every analysis of events has to detect them anew, and
> the detection channels cannot be matched to the paper's.

tingley-2019-02 reports what remains of this in its finding 8 (recordings without an event table).

In [ ]:
has_ls = [m for m in M if "LSRipples" in m["events"]]
has_ca1 = [m for m in M if "CA1Ripples" in m["events"]]
print(f"files with LSRipples: {len(has_ls)} ({sum(len(m['events']['LSRipples']) for m in has_ls)} events); "
      f"with CA1Ripples: {len(has_ca1)} ({sum(len(m['events']['CA1Ripples']) for m in has_ca1)} events)")
print("files whose LSRipples description names the detection channel and the stored parameters:",
      sum("Detected on channel" in m["event_desc"]["LSRipples"] and "Detection parameters" in m["event_desc"]["LSRipples"] for m in has_ls))
print("files with a channel marked in ripple_detection_channel:", sum(any(m["det"]) for m in M))
print("files with no event table at all:", [f"{m['subject']} {m['session']}" for m in M if not m["events"]])
print("files with LS units and no LSRipples:", [f"{m['subject']} {m['session']}" for m in M if (m["unit_region"] == "ls").any() and "LSRipples" not in m["events"]])

## Finding 2

> Nine files (DT5 2016-10-07, DT7 2017-03-14, -16, -17, -19, 2017-04-07, -10, DT9 2017-04-28, 2017-05-26) store
> units without a location column, so their 268 units cannot be assigned to LS or hippocampus.

tingley-2019-02 reports what remains of this in its finding 6.

In [ ]:
nine = ["20161007-720um", "20170314-72um", "20170316-288um", "20170317-288um", "20170319-288um", "20170407-936um",
        "20170410-1152um", "20170428-0um", "20170526-1116um"]
for m in M:
    if m["session"].startswith(tuple(nine)):
        print(f"{m['subject']} {m['session']}: {m['n_units']} units, regions {dict(collections.Counter(m['unit_region']))}")
print("files with a unit whose region is empty:", [(m["session"], int((m["unit_region"] == "").sum())) for m in M if (m["unit_region"] == "").any()])

## Finding 3

> Hippocampal electrodes and units are labelled `hpc` or `ca3`; `hpc` does not say whether a site is in CA1,
> which the paper's analyses distinguish. Four DT7 files label all electrodes `unknown`.

tingley-2019-02 reports the first part in its finding 5. The second part is checked here.

In [ ]:
print("electrode regions:", dict(collections.Counter(r for m in M for r in m["el_region"])))
print("unit regions:", dict(collections.Counter(r for m in M for r in m["unit_region"])))
print("files whose electrodes are all 'unknown':")
for m in M:
    if (m["el_region"] == "unknown").all():
        print(f"  {m['subject']} {m['session']}: {len(m['el_region'])} electrodes, {m['n_units']} units, event tables {sorted(m['events'])}")
print("other files with 'unknown' electrodes:", [(m["session"], int((m["el_region"] == "unknown").sum())) for m in M
                                                 if (m["el_region"] == "unknown").any() and not (m["el_region"] == "unknown").all()])

## Finding 4

> The paper's CA3 recordings used a CA1 reference electrode to detect ripples. The 10 files with `ca3`
> electrodes hold 64 `ca3` and 32 `ls` channels and no CA1 channel, and the paper reports 25 CA3 and LS
> recordings against 10 here.

tingley-2019-02 reports what remains of this in its finding 8 (and the count of CA3 recordings in its finding 5).

In [ ]:
rows = []
for m in M:
    if (m["el_region"] == "ca3").any():
        rows.append(dict(subject=m["subject"], session=m["session"][:32], electrodes=dict(collections.Counter(m["el_region"])),
                         ca1_flag_regions=list(m["el_region"][m["ca1_flag"]]), CA1Ripples="CA1Ripples" in m["events"]))
print(pd.DataFrame(rows).to_string())

## Finding 5

> Electrode x, y and z are NaN in every file. The probe geometry and its orientation in the brain, needed for
> the travelling-wave analysis of Figure 3, are not given, and the relative depth of each recording within LS
> (Figure 2) is not given.

tingley-2019-02 reports this in its findings 2 and 3.

In [ ]:
print("electrode table columns (files):", dict(collections.Counter(tuple(m["el_cols"]) for m in M)))
print("files with x, y or z:", sum(bool({"x", "y", "z"} & set(m["el_cols"])) for m in M))

## Finding 6

> The units carry no waveforms, no channel and no quality metric (the Methods compute isolation distance), so
> the CA1 depth of each neuron (Figure 7) cannot be estimated and single units cannot be told from multi-unit
> clusters; 11 units fire above 50 Hz on average. Wide-band data, from which waveforms could be re-extracted, are
> in 27 of the 98 files (DT1, DT2 and two DT9 files); 1579 of 2778 hippocampal units have none.

tingley-2019-02 reports what remains of this in its finding 16.

In [ ]:
U = pd.DataFrame([dict(subject=m["subject"], units=m["n_units"], waveform=m["wf_units"], isolation=m["iso_units"],
                       files=1, files_wf=int(m["wf_units"] > 0)) for m in M if m["n_units"]])
print("unit columns (files):", dict(collections.Counter(c for m in M for c in m["unit_cols"])))
print("units with a finite raw_waveform and with an isolation distance, by subject:")
print(U.groupby("subject").sum().to_string())
print("total:", U[["units", "waveform", "isolation"]].sum().to_dict())

## Finding 7

> The two rats of the optogenetic experiments (Figure 6) are not in the dandiset.

tingley-2019-02 reports this in its finding 1.

In [ ]:
print("subjects:", dict(collections.Counter(m["nwb_subject"] for m in M)))

## Finding 8

> 25 units in 10 files have spike times out of order (1 to 2782 inversions each), against the NWB rule that
> spike times are sorted.

Checked here on the spike times as stored (the entry's helper sorts them when it loads them, so its own check
cannot see this).

In [ ]:
inv = [(m["subject"], m["session"], i, n) for m in M for i, n in enumerate(m["inversions"]) if n > 0]
print(f"units: {sum(len(m['inversions']) for m in M)}; units with spike times out of order: {len(inv)}")
for r in inv: print("  ", r)
said = [(m["subject"], m["session"]) for m in M if "not in increasing order in the source" in m["units_desc"]]
print(f"files whose units description says spike times were out of order in the source and are sorted: {len(said)}")
for r in said: print("  ", r)

## Finding 9

> The paper's Key Resources point to github.com/DavidTingley/papers for scripts that recreate each main figure,
> but the repository holds only the network model of Figure 8 and the code of the 2018 paper; the analyses of
> Figures 1 to 7 are not published.

This concerns the paper's code, not the dataset; the repository is checked as it is today.

In [ ]:
api = "https://api.github.com/repos/DavidTingley/papers"
head = json.load(urllib.request.urlopen(f"{api}/commits?per_page=1"))[0]
tree = json.load(urllib.request.urlopen(f"{api}/git/trees/{head['sha']}?recursive=1"))["tree"]
files = [t["path"] for t in tree if t["type"] == "blob"]
print("head commit:", head["sha"][:7], head["commit"]["committer"]["date"])
print(f"{len(files)} files; top-level folders:", dict(collections.Counter(p.split("/")[0] for p in files)))
print("MATLAB scripts:", [p for p in files if p.endswith(".m")])
print("files naming 2019, ripples, HFOs or the septum:", [p for p in files if any(k in p.lower() for k in ("2019", "ripple", "hfo", "septum"))])

## Finding 10

> The HFO band is given as 120-180 Hz in the Summary and the legends of Figures 1, 2 and 5 and as 120-200 Hz in
> the Results and Methods. The Methods call the 62-recording dataset that of Figure 6, which shows optogenetics,
> where Figure 7 is meant, and cite Figure 3A and 3B (travelling waves) for the jitter control and the state of
> phase-locking epochs, which appear to be Figure S3A and Figure 5.

This concerns the paper, which has not changed; it cannot be checked on the dataset. It was checked against the
paper's text (the accepted manuscript, PMC6952543), which is not part of this entry: the Summary and the legends
of Figures 1, 2 and 5 still give 120–180 Hz, the Results ("phase-locked to the 120–200 Hz frequency band") and
the Methods (ripple detection, phase locking) 120–200 Hz; the Methods still say "The dataset used for Figure 6
consists of 62 recordings", and still cite Figure 3B for the phase-locking epochs and Figure 3A for the jitter
control of the excess correlation.

## Finding 11

> In this attempt's detection, 35% of LS HFOs within 50 ms of a CA1 ripple fall on the same 0.8 ms sample as
> the ripple (71% in DT8, 62% in DT2, 7% in DT9), and the CA1-LS 120-200 Hz phase difference is consistent even
> at random times (median phase-locking value 0.134 against about 0.021 for random phases). Re-referencing the
> LS channel to another LS channel on its shank lowers the same-sample share to 4.1%. This points to a signal
> common to the LS and hippocampal channels, such as the reference, which the files do not document; some files
> list a device of "reference sites a few millimeters dorsal" without saying which channels use it.

Two checks on this version. (a) The same-sample share for the lab's deposited detections (LSRipples and
CA1Ripples, peak times), as the earlier attempt measured it on its own: for each LS HFO the nearest CA1 ripple
within 50 ms, and the fraction of those pairs less than 0.5 ms apart. (b) The CA1-LS phase relation at random
times, on the lab's two detection channels of each recording with both tables: 2000 random times in the
second 16-channel storage chunk of the recording (about 840 s), the 120-200 Hz phase (4th order Butterworth,
zero phase, Hilbert) of each channel, and the phase-locking value of their difference; as a control, the same
with the CA1 phase taken 1 s later. The LFP is the same data as in the earlier version (the files' LFP
description says so).

In [ ]:
def nearest_lag(a, b):
    j = np.clip(np.searchsorted(b, a), 1, len(b) - 1)
    x, y = a - b[j - 1], a - b[j]
    return np.where(np.abs(x) < np.abs(y), x, y)
both = [m for m in M if "LSRipples" in m["events"] and "CA1Ripples" in m["events"]]
lags = {m["session"]: nearest_lag(m["events"]["LSRipples"], m["events"]["CA1Ripples"]) * 1000 for m in both}
lags = {k: v[np.abs(v) <= 50] for k, v in lags.items()}
allv = np.concatenate(list(lags.values()))
print(f"(a) {len(both)} recordings with LSRipples and CA1Ripples; {len(allv)} LS HFOs with a CA1 ripple within 50 ms; "
      f"less than 0.5 ms apart: {np.mean(np.abs(allv) < 0.5):.3f}")
for s in sorted({m["subject"] for m in both}):
    v = np.concatenate([lags[m["session"]] for m in both if m["subject"] == s])
    print(f"   {s}: {len(v)} pairs, same sample {np.mean(np.abs(v) < 0.5):.3f}")
print("   DT8 files with CA1Ripples:", sum(m["subject"] == "DT8" and "CA1Ripples" in m["events"] for m in M))

In [ ]:
def det_row(m, name):
    r = [i for i, d in enumerate(m["det"]) if name in d.split(",") or name in d]
    return r[0] if r else None
P = [m for m in both if det_row(m, "LSRipples") is not None and det_row(m, "CA1Ripples") is not None]
SEED = zlib.crc32(KEY.encode())
def plv_job(a):
    i, m = a
    rng = np.random.default_rng([SEED, i])
    ls, ca = det_row(m, "LSRipples"), det_row(m, "CA1Ripples")
    X, _, _ = T.read_lfp_block(m["url"], ls, [ls], t_chunks=[1]); Y, _, _ = T.read_lfp_block(m["url"], ca, [ca], t_chunks=[1])
    fx, fy = T.bandpass(X[:, 0].astype(np.float32), 120, 200, 4), T.bandpass(Y[:, 0].astype(np.float32), 120, 200, 4)
    px, py = np.angle(T.analytic(fx)), np.angle(T.analytic(fy))
    k = rng.integers(int(T.FS_LFP), len(fx) - int(T.FS_LFP), 2000)
    return dict(subject=m["subject"], session=m["session"][:30], ls_region=m["el_region"][ls], ca1_region=m["el_region"][ca],
                plv=float(np.abs(np.mean(np.exp(1j * (py[k] - px[k]))))),
                plv_ca1_1s_later=float(np.abs(np.mean(np.exp(1j * (py[k + int(T.FS_LFP)] - px[k]))))),
                band_corr=float(np.corrcoef(fx, fy)[0, 1]))
t0 = time.time()
with ThreadPoolExecutor(4) as ex:
    PL = pd.DataFrame(list(ex.map(plv_job, enumerate(P))))
print(f"(b) {len(PL)} recordings, {time.time() - t0:.0f} s; detection channel regions LS {dict(collections.Counter(PL.ls_region))}, CA1 {dict(collections.Counter(PL.ca1_region))}")
print(PL[["plv", "plv_ca1_1s_later", "band_corr"]].describe().loc[["count", "min", "50%", "max"]].round(3).to_string())
print("by subject (median):"); print(PL.groupby("subject")[["plv", "plv_ca1_1s_later", "band_corr"]].median().round(3).to_string())
print(f"phase-locking value expected for 2000 random phases: about {np.sqrt(np.pi) / 2 / np.sqrt(2000):.3f}")

## Summary

Each section's result is stated in the reconciliation block of `attempts/tingley-2019-02.yaml`.